In [1]:
import geopandas as gpd
import pandas as pd

In [20]:
lakes_gdf = gpd.read_file('model_data\shapefiles\glacial_lakes\glacial_lake_single_polygon\glacial_lake_single polygon.shp')
lakes_gdf.columns

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
C:\Users\rubel\AppData\Local\Temp\ipykernel_20992\757817109.py:1: SyntaxWarning: invalid escape sequence '\s'
  lakes_gdf = gpd.read_file('model_data\shapefiles\glacial_lakes\glacial_lake_single_polygon\glacial_lake_single polygon.shp')


Index(['fid', 'Type', 'Elevation', 'GLAKE_ID', 'area_2020', 'perimeter_',
       'area_1990', 'perimete_1', 'geometry_1', 'area_2000', 'perimete_2',
       'geometry_2', 'area_2010', 'perimete_3', 'geometry_3', 'area_2015',
       'perimete_4', 'geometry_4', 'area_1990_', 'area_2000_', 'area_2010_',
       'area_2015_', 'area_2020_', 'expansion_', 'expansio_1', 'glof_happe',
       'glof_count', 'lake_type', 'geometry'],
      dtype='str')

In [23]:
print(lakes_gdf.iloc[6473])

fid                                                      6473.0
Type                                  Unconnected glacial lakes
Elevation                                                4127.0
GLAKE_ID                                        GL099549E30494N
area_2020                                        1068871.924268
perimeter_                                          4195.545967
area_1990                                        1050468.549542
perimete_1                                          4144.583373
geometry_1    POLYGON ((406663.339112 67535.348732, 406593.8...
area_2000                                        1083895.619119
perimete_2                                          4218.762547
geometry_2    POLYGON ((406682.504169 67519.920554, 406544.2...
area_2010                                        1054243.544303
perimete_3                                          4193.780303
geometry_3    POLYGON ((406669.002269 67536.078983, 406624.2...
area_2015                               

In [4]:
glaciers_gdf = gpd.read_file('model_data\shapefiles\glacier\glaciers_in_aoi.gpkg')

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
C:\Users\rubel\AppData\Local\Temp\ipykernel_20992\2000784639.py:1: SyntaxWarning: invalid escape sequence '\s'
  glaciers_gdf = gpd.read_file('model_data\shapefiles\glacier\glaciers_in_aoi.gpkg')


In [8]:
from pysheds.grid import Grid

In [37]:
import math
import numpy as np
import geopandas as gpd
import xarray as xr
from shapely.geometry import Point, shape
from pystac_client import Client
import planetary_computer
import odc.stac
import rioxarray
from pysheds.grid import Grid
import rasterio.features
import tempfile
import os

def get_utm_epsg(lon, lat):
    """Dynamically calculates the correct WGS84 UTM EPSG code."""
    utm_zone = int((lon + 180) / 6) + 1
    return 32600 + utm_zone

def find_upstream_glacier_for_unconnected_lake(lake_row, glaciers_gdf, catalog):
    """
    Complete pipeline to link an unconnected lake to its closest upstream glacier.

    Parameters:
    - lake_row: A single row (Series) from a GeoDataFrame of lakes (must be EPSG:4326).
    - glaciers_gdf: GeoDataFrame containing all regional glaciers (must be EPSG:4326).
    - catalog: Initialized pystac_client for Planetary Computer.

    Returns:
    - A dictionary containing the closest glacier's index, geometry, and distance in meters.
    """

    # --- Step 0: Filter Check ---
    # Only process lakes classified as "Unconnected glacial lakes"
    if lake_row.get('Type', '') != 'Unconnected glacial lakes':
        return {"status": "skipped", "reason": "Lake is not classified as unconnected"}

    lake_geom = lake_row.geometry

    # --- Step 1: Stream DEM via STAC ---
    bbox = lake_geom.buffer(0.30).bounds  # ~15km buffer
    search = catalog.search(collections=["cop-dem-glo-30"], bbox=bbox)
    items = list(search.items())

    if not items:
        return {"status": "error", "reason": "No DEM data found."}

    dem_dataset = odc.stac.load(
        items,
        bbox=bbox,
        crs="EPSG:4326",
        resolution=0.0002777777777777778,
        patch_url=planetary_computer.sign,
    )

    dem_array = dem_dataset["data"].squeeze()
    if 'longitude' in dem_array.dims and 'latitude' in dem_array.dims:
        dem_array = dem_array.rename({'longitude': 'x', 'latitude': 'y'})
    dem_array = dem_array.rio.write_crs("EPSG:4326")

    # --- Step 2: Find the Pour Point (Lowest Elevation on Boundary) ---
    lon_pts, lat_pts = lake_geom.exterior.xy
    lons_da = xr.DataArray(lon_pts, dims="points")
    lats_da = xr.DataArray(lat_pts, dims="points")

    elevations = dem_array.sel(x=lons_da, y=lats_da, method="nearest").values
    min_idx = np.nanargmin(elevations)
    pour_lon, pour_lat = lon_pts[min_idx], lat_pts[min_idx]

    # --- Step 3: Reproject to UTM for Hydrology ---
    utm_epsg = get_utm_epsg(pour_lon, pour_lat)
    utm_crs = f"EPSG:{utm_epsg}"

    dem_utm = dem_array.rio.reproject(utm_crs)
    pour_point_utm = gpd.GeoSeries([Point(pour_lon, pour_lat)], crs="EPSG:4326").to_crs(utm_crs).iloc[0]

    # --- Step 4: Delineate Catchment using Pysheds ---
    tmp = tempfile.NamedTemporaryFile(suffix=".tif", delete=False)
    tmp_path = tmp.name
    tmp.close()

    try:
        dem_utm.rio.to_raster(tmp_path)
        grid = Grid.from_raster(tmp_path)
        dem = grid.read_raster(tmp_path)

        pit_filled = grid.fill_pits(dem)
        flooded = grid.resolve_flats(pit_filled)
        fdir = grid.flowdir(flooded)
        catchment_grid = grid.catchment(
            x=pour_point_utm.x,
            y=pour_point_utm.y,
            fdir=fdir,
            xytype='coordinate'
        )

    finally:
        if os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except PermissionError:
                pass

    # Convert raster catchment mask to polygon geometry
    # Convert raster catchment mask to polygon geometry
    catchment_array = np.asarray(catchment_grid) # <-- YOU NEED THIS LINE
    shapes = rasterio.features.shapes(
        catchment_array.astype(np.int16),
        mask=(catchment_array == 1),
        transform=grid.affine,
    )
    catchment_polygons = [shape(geom) for geom, value in shapes if value == 1]

    if not catchment_polygons:
        return {"status": "error", "reason": "Catchment delineation failed."}

    main_catchment = max(catchment_polygons, key=lambda a: a.area)
    catchment_gdf = gpd.GeoDataFrame(geometry=[main_catchment], crs=utm_crs)

    # --- Step 5: Spatial Intersection & Distance Calculation ---
    # NEW CODE: Add a 100-meter tolerance buffer to account for DEM/RGI misalignment
    catchment_buffered = catchment_gdf.buffer(100) # Buffers by 100 meters in UTM
    catchment_4326 = catchment_buffered.to_crs("EPSG:4326").geometry.iloc[0]

    upstream_glaciers = glaciers_gdf[glaciers_gdf.intersects(catchment_4326)].copy()

    if upstream_glaciers.empty:
        return {
            "status": "success",
            "glacier_id": None,
            "distance_m": None,
            "note": "No glaciers in upstream catchment.",
        }

    lake_utm = gpd.GeoSeries([lake_geom], crs="EPSG:4326").to_crs(utm_crs).iloc[0]
    upstream_glaciers_utm = upstream_glaciers.to_crs(utm_crs)
    upstream_glaciers_utm['distance_to_lake'] = upstream_glaciers_utm.distance(lake_utm)

    closest_glacier = upstream_glaciers_utm.loc[upstream_glaciers_utm['distance_to_lake'].idxmin()]

    return {
        "status": "success",
        "glacier_index": closest_glacier.name,
        "distance_m": closest_glacier['distance_to_lake'],
    }

In [40]:
import os
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
from pystac_client import Client
import planetary_computer

# Assume 'find_upstream_glacier_for_unconnected_lake' is defined above

import os
import time
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
from pystac_client import Client
import planetary_computer

def process_lakes_with_backoff(lakes_gdf, glaciers_gdf, output_csv, max_retries=5, base_delay=5):
    """
    Processes lakes with automatic checkpointing and an exponential backoff retry loop.
    """
    catalog = Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace
    )
    
    processed_ids = set()
    if os.path.exists(output_csv):
        try:
            existing_df = pd.read_csv(output_csv)
            processed_ids = set(existing_df['lake_id'].dropna().astype(str))
            print(f"Resuming from checkpoint: {len(processed_ids)} lakes already processed.")
        except pd.errors.EmptyDataError:
            pass
            
    write_header = not os.path.exists(output_csv)
    
    with open(output_csv, 'a') as f:
        if write_header:
            f.write("lake_id,status,glacier_index,distance_m,error_reason\n")
            
        for index, lake_row in tqdm(lakes_gdf.iterrows(), total=len(lakes_gdf), desc="Processing Lakes"):
            lake_id = str(lake_row.get('lake_id', index))
            
            if lake_id in processed_ids:
                continue
            
            # --- Exponential Backoff Retry Loop ---
            success = False
            for attempt in range(max_retries):
                try:
                    result = find_upstream_glacier_for_unconnected_lake(lake_row, glaciers_gdf, catalog)
                    
                    status = result.get('status', 'unknown')
                    g_idx = result.get('glacier_index', '')
                    dist = result.get('distance_m', '')
                    reason = result.get('reason', '')
                    
                    f.write(f"{lake_id},{status},{g_idx},{dist},{reason}\n")
                    f.flush() 
                    success = True
                    break # Exit the retry loop on success
                    
                except Exception as e:
                    error_msg = str(e)
                    # Check if it is a network/API related error
                    if any(x in error_msg for x in ["429", "502", "503", "timeout", "Connection"]):
                        delay = base_delay * (2 ** attempt) # Delays: 5s, 10s, 20s, 40s, 80s
                        tqdm.write(f"Network error on lake {lake_id}. Retrying in {delay}s... (Attempt {attempt + 1}/{max_retries})")
                        time.sleep(delay)
                    else:
                        # If it's a structural error (e.g., geometry issue), log it and don't retry
                        f.write(f"{lake_id},error,,,{error_msg.replace(',', ';')}\n")
                        f.flush()
                        success = True 
                        break
            
            # If all retries are exhausted and it still failed
            if not success:
                f.write(f"{lake_id},error,,,Max API retries exceeded\n")
                f.flush()
# --- Execution Example ---
# Assuming lakes_gdf and rgi_glaciers_gdf are already loaded in memory
# Force both GeoDataFrames into geographic coordinates (degrees)
lakes_gdf = lakes_gdf.to_crs("EPSG:4326")
glaciers_gdf = glaciers_gdf.to_crs("EPSG:4326")

# Now run the batch processing function
process_lakes_with_backoff(lakes_gdf, glaciers_gdf, "hma_lake_glacier_linkage1.csv")
# process_lakes_resiliently(lakes_gdf,glaciers_gdf, "hma_lake_glacier_linkage.csv")

Processing Lakes: 100%|██████████| 6924/6924 [14:02:20<00:00,  7.30s/it]    


In [ ]:
glake_single = pd.read_csv('hma_lake_glacier_linkage1.csv')
link2 = pd.read_csv('hma_lake_glacier_linkage2.csv')


In [47]:
link2['distance_m'].value_counts()

distance_m
0.000000       174
432.792479       1
285.006357       1
1617.363986      1
563.655012       1
              ... 
130.814685       1
51.967509        1
650.352670       1
105.117227       1
6.520819         1
Name: count, Length: 651, dtype: int64

In [54]:
glake = gpd.read_file(r'model_data\shapefiles\glacial_lakes\glake_final.gpkg')
glake_single = gpd.read_file(r'model_data\shapefiles\glacial_lakes\glacial_lake_single_polygon\glacial_lake_single polygon.shp')

In [59]:
# Check repeated lake_id values in link1
import pandas as pd

# Fallback in case link1 is not defined but glake_single exists
if 'link1' not in globals():
    if 'glake_single' in globals():
        link1 = glake_single.copy()
        print("'link1' not found. Using 'glake_single' as fallback.")
    else:
        raise NameError("Neither 'link1' nor 'glake_single' is defined.")

def detect_id_column(df):
    cols = list(df.columns)

    # Prefer exact match
    for c in ["lake_id", "Lake_ID", "LAKE_ID", "id", "ID", "fid", "FID"]:
        if c in cols:
            return c

    # Fallback: any column containing lake_id or id
    id_like = [c for c in cols if "lake_id" in c.lower()]
    if id_like:
        return id_like[0]

    id_like = [c for c in cols if c.lower() in ["id", "fid"] or "id" in c.lower()]
    if id_like:
        return id_like[0]

    return None

id_col = detect_id_column(link2)
print("Detected ID column in link1:", id_col)

if id_col is None:
    print("No ID-like column found in link1.")
else:
    ids = link1[id_col].astype(str).str.strip()

    dup_mask = ids.duplicated(keep=False)
    dup_rows = link1.loc[dup_mask].copy()

    dup_counts = (
        ids[dup_mask]
        .value_counts()
        .rename_axis(id_col)
        .reset_index(name="repeat_count")
        .sort_values("repeat_count", ascending=False)
    )

    print("Total rows:", len(link1))
    print("Unique IDs:", ids.nunique(dropna=True))
    print("Number of repeated ID values:", len(dup_counts))
    print("Total rows involved in repeats:", len(dup_rows))

    if dup_counts.empty:
        print("No repeated lake_id values found in link1.")
    else:
        print("Repeated lake_id values and counts:")
        display(dup_counts)

        print("Rows with repeated lake_id values:")
        cols_to_show = [id_col] + [c for c in ["status", "glacier_index", "distance_m", "error_reason", "Type", "geometry"] if c in dup_rows.columns]
        display(dup_rows[cols_to_show].sort_values(by=id_col))

Detected ID column in link1: lake_id
Total rows: 6924
Unique IDs: 6924
Number of repeated ID values: 0
Total rows involved in repeats: 0
No repeated lake_id values found in link1.


In [70]:
# Distance from each glake feature to its nearest glacier (in meters)
# and nearest glacier RGI ID
import numpy as np
import geopandas as gpd

# Validate required dataframes
if 'glake' not in globals():
    raise NameError("glake is not defined. Load/create glake first.")
if 'glaciers_gdf' not in globals():
    raise NameError("glaciers_gdf is not defined. Load/create glaciers_gdf first.")

# Ensure both layers have CRS
if glake.crs is None or glaciers_gdf.crs is None:
    raise ValueError("Both glake and glaciers_gdf must have valid CRS.")

# Detect glacier ID column (prefer RGIId)
rgi_candidates = [c for c in glaciers_gdf.columns if c.lower() == 'rgiid']
if not rgi_candidates:
    rgi_candidates = [c for c in glaciers_gdf.columns if 'rgi' in c.lower() and 'id' in c.lower()]
if not rgi_candidates:
    raise ValueError("Could not find an RGIId-like column in glaciers_gdf.")
rgi_col = rgi_candidates[0]

# Choose a projected CRS for metric distances
if glake.crs.to_string() != "EPSG:4326":
    glake_wgs84 = glake.to_crs("EPSG:4326")
else:
    glake_wgs84 = glake.copy()

center = glake_wgs84.geometry.unary_union.centroid
if 'get_utm_epsg' in globals():
    metric_epsg = get_utm_epsg(center.x, center.y)
    metric_crs = f"EPSG:{metric_epsg}"
else:
    metric_crs = "EPSG:3857"

# Reproject for distance calculation
glake_m = glake.to_crs(metric_crs).copy()
glaciers_m = glaciers_gdf.to_crs(metric_crs).copy()

# Stable row id for safe assignment
left = glake_m.reset_index(drop=True).copy()
left['_row_id'] = np.arange(len(left))

# Keep geometry and RGIId from glaciers
right = glaciers_m[[rgi_col, 'geometry']].copy()

nearest = gpd.sjoin_nearest(
    left[['_row_id', 'geometry']],
    right,
    how='left',
    distance_col='distance_to_glacier_m'
)

# Resolve tie cases (multiple nearest glaciers at same distance)
nearest_sorted = nearest.sort_values(by=['_row_id', 'distance_to_glacier_m', rgi_col], kind='mergesort')
best = nearest_sorted.drop_duplicates(subset=['_row_id'], keep='first').set_index('_row_id')

# Assign results back to glake by row position
glake['distance_to_glacier_m'] = pd.Series(best['distance_to_glacier_m']).reindex(range(len(glake))).values
glake['nearest_rgiid'] = pd.Series(best[rgi_col]).reindex(range(len(glake))).values

print(f"Added columns: distance_to_glacier_m, nearest_rgiid (from '{rgi_col}')")
print(glake[['distance_to_glacier_m', 'nearest_rgiid']].head())

C:\Users\rubel\AppData\Local\Temp\ipykernel_20992\3276941113.py:30: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = glake_wgs84.geometry.unary_union.centroid


Added columns: distance_to_glacier_m, nearest_rgiid (from 'RGIId')
   distance_to_glacier_m   nearest_rgiid
0               8.439836  RGI60-14.07524
1               0.000000  RGI60-14.09122
2               0.000000  RGI60-14.09057
3               0.000000  RGI60-14.09028
4               0.000000  RGI60-14.08932


In [64]:
# To see the index (which usually matches the fid)
print(glake.index)

# To turn the index into a physical 'fid' column like you see in QGIS
# gdf = gdf.reset_index().rename(columns={'index': 'fid'})

RangeIndex(start=0, stop=6923, step=1)


In [69]:
glaciers_gdf.columns

Index(['RGIId', 'GLIMSId', 'BgnDate', 'EndDate', 'CenLon', 'CenLat',
       'O1Region', 'O2Region', 'Area', 'Zmin', 'Zmax', 'Zmed', 'Slope',
       'Aspect', 'Lmax', 'Status', 'Connect', 'Form', 'TermType', 'Surging',
       'Linkages', 'Name', 'geometry'],
      dtype='str')

In [63]:
glake.columns

Index(['Type', 'Elevation', 'GLAKE_ID', 'area_2020', 'perimeter_2020',
       'area_1990', 'perimeter_1990', 'geometry_1990', 'area_2000',
       'perimeter_2000', 'geometry_2000', 'area_2010', 'perimeter_2010',
       'geometry_2010', 'area_2015', 'perimeter_2015', 'geometry_2015',
       'area_1990_km2', 'area_2000_km2', 'area_2010_km2', 'area_2015_km2',
       'area_2020_km2', 'expansion_rate_km2_yr', 'expansion_rate_pct_yr',
       'glof_happened', 'glof_count', 'lake_type', 'geometry'],
      dtype='str')

In [83]:
glake.to_file(r'model_data\shapefiles\glacial_lakes\glake_trial_distance.gpkg', driver='GPKG')

In [73]:
# Add connectivity flag based on nearest-glacier distance
# 1 = connected (distance == 0), 0 = not connected
import numpy as np

if 'glake' not in globals():
    raise NameError("glake is not defined.")
if 'distance_to_glacier_m' not in glake.columns:
    raise KeyError("Column 'distance_to_glacier_m' not found in glake. Run nearest-distance cell first.")

dist = pd.to_numeric(glake['distance_to_glacier_m'], errors='coerce')
glake['is_connected'] = np.where(np.isclose(dist, 0.0, atol=1e-9), 1, 0).astype('int64')

print("Added column: is_connected")
print(glake['is_connected'].value_counts(dropna=False))

Added column: is_connected
is_connected
0    5937
1     986
Name: count, dtype: int64


In [75]:
dam = gpd.read_file(r'dam_characteristics_2.csv')

In [76]:
dam['.geo']

0       {"type":"Polygon","coordinates":[[[77.17606057...
1       {"type":"Polygon","coordinates":[[[73.87725685...
2       {"type":"Polygon","coordinates":[[[91.80517578...
3       {"type":"Polygon","coordinates":[[[74.87687607...
4       {"type":"Polygon","coordinates":[[[86.06279253...
                              ...                        
6918    {"type":"Polygon","coordinates":[[[99.61444672...
6919    {"type":"Polygon","coordinates":[[[97.79569430...
6920    {"type":"Polygon","coordinates":[[[88.71124008...
6921    {"type":"Polygon","coordinates":[[[95.86312140...
6922    {"type":"Polygon","coordinates":[[[95.53891593...
Name: .geo, Length: 6923, dtype: str

In [77]:
# Convert dam table .geo column to proper geometry and save for QGIS
import os
import json
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape

# Use existing table if present; otherwise read from CSV
if 'dam_df' in globals():
    dam_table = dam_df.copy()
elif 'dam' in globals() and isinstance(dam, pd.DataFrame):
    dam_table = dam.copy()
else:
    dam_csv = 'dam_characteristics_hma.csv'
    dam_table = pd.read_csv(dam_csv)
    print(f"Loaded dam table from: {dam_csv}")

# Detect the source geometry column
geo_src_col = '.geo' if '.geo' in dam_table.columns else ('geo' if 'geo' in dam_table.columns else None)
if geo_src_col is None:
    raise KeyError("No '.geo' or 'geo' column found in dam table.")

def parse_geojson_geometry(value):
    if pd.isna(value):
        return None

    obj = value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        obj = json.loads(text)

    # Handle GeoJSON Feature objects
    if isinstance(obj, dict) and obj.get('type') == 'Feature' and 'geometry' in obj:
        obj = obj['geometry']

    if isinstance(obj, dict) and 'type' in obj and 'coordinates' in obj:
        return shape(obj)

    return None

geom = dam_table[geo_src_col].apply(parse_geojson_geometry)

dam_gdf = gpd.GeoDataFrame(
    dam_table.drop(columns=[geo_src_col]),
    geometry=geom,
    crs='EPSG:4326'
)

# Keep only valid, non-empty geometries
valid_mask = dam_gdf.geometry.notna() & (~dam_gdf.geometry.is_empty)
dam_gdf = dam_gdf.loc[valid_mask].copy()

out_gpkg = r'model_data\shapefiles\dam\dam_characteristics_hma.gpkg'
os.makedirs(os.path.dirname(out_gpkg), exist_ok=True)
dam_gdf.to_file(out_gpkg, layer='dam_characteristics_hma', driver='GPKG', index=False)

print(f"Saved {len(dam_gdf)} features with proper geometry to: {out_gpkg}")

Saved 6922 features with proper geometry to: model_data\shapefiles\dam\dam_characteristics_hma.gpkg


In [ ]:
dam_gdf.head()

In [79]:
# Match glake and dam_gdf by geometry and transfer dam attributes to glake
import geopandas as gpd

if 'glake' not in globals():
    raise NameError("glake is not defined.")
if 'dam_gdf' not in globals():
    raise NameError("dam_gdf is not defined. Run the dam geometry conversion cell first.")

if glake.crs is None or dam_gdf.crs is None:
    raise ValueError("Both glake and dam_gdf must have valid CRS.")

# Align CRS for reliable geometry matching
if glake.crs != dam_gdf.crs:
    dam_match = dam_gdf.to_crs(glake.crs).copy()
else:
    dam_match = dam_gdf.copy()

glake_match = glake.copy()

# Build stable geometry keys (exact topology match)
glake_match['_geom_key'] = glake_match.geometry.normalize().to_wkb(hex=True)
dam_match['_geom_key'] = dam_match.geometry.normalize().to_wkb(hex=True)

# Keep all dam attributes except geometry and excluded field(s)
exclude_cols = {'geometry', 'system:index', '_geom_key'}
dam_attr_cols = [c for c in dam_match.columns if c not in exclude_cols]

# If duplicate geometry in dam_gdf, keep first row to avoid one-to-many explosion
dam_attr = dam_match[['_geom_key'] + dam_attr_cols].drop_duplicates(subset=['_geom_key'], keep='first')

# Handle column name collisions by renaming incoming dam columns
existing = set(glake_match.columns)
rename_map = {}
for c in dam_attr_cols:
    if c in existing:
        rename_map[c] = f"{c}_dam"
if rename_map:
    dam_attr = dam_attr.rename(columns=rename_map)

# Left join attributes into glake based on geometry key
joined = glake_match.merge(dam_attr, on='_geom_key', how='left')

# Count matches for quick QA
added_cols = [c for c in dam_attr.columns if c != '_geom_key']
if added_cols:
    matched_rows = joined[added_cols].notna().any(axis=1).sum()
else:
    matched_rows = 0

# Clean temporary key and write back
glake = gpd.GeoDataFrame(joined.drop(columns=['_geom_key']), geometry='geometry', crs=glake.crs)

print(f"Matched geometries and added {len(added_cols)} attribute column(s) from dam_gdf to glake.")
print(f"Rows in glake with at least one matched dam attribute: {matched_rows} / {len(glake)}")
print("Added columns:", added_cols)

Matched geometries and added 6 attribute column(s) from dam_gdf to glake.
Rows in glake with at least one matched dam attribute: 0 / 6923
Added columns: ['dam_crest_m', 'dam_height_m', 'dam_slope_deg', 'dam_width_m', 'freeboard_m', 'wse_m']


In [82]:
# Why NaN happened + robust attribute transfer from dam_gdf to glake
import geopandas as gpd
import numpy as np
import pandas as pd

if 'glake' not in globals() or 'dam_gdf' not in globals():
    raise NameError("glake and dam_gdf must be defined.")

if glake.crs is None or dam_gdf.crs is None:
    raise ValueError("Both glake and dam_gdf must have valid CRS.")

# 1) Align CRS
if glake.crs != dam_gdf.crs:
    dam_match = dam_gdf.to_crs(glake.crs).copy()
else:
    dam_match = dam_gdf.copy()

# 2) Quick diagnostics
left_key = glake.geometry.normalize().to_wkb(hex=True)
right_key = dam_match.geometry.normalize().to_wkb(hex=True)
exact_matches = len(set(left_key) & set(right_key))
print(f"Exact geometry-key matches: {exact_matches}")

intersections_count = len(gpd.sjoin(glake[['geometry']], dam_match[['geometry']], how='inner', predicate='intersects'))
print(f"Spatial intersects matches: {intersections_count}")

# 3) Transfer all dam attributes except excluded fields
exclude_cols = {'geometry', 'system:index'}
dam_attr_cols = [c for c in dam_match.columns if c not in exclude_cols]

# Avoid name collisions in glake
rename_map = {}
for c in dam_attr_cols:
    if c in glake.columns:
        rename_map[c] = f"{c}_dam"
dam_join = dam_match.rename(columns=rename_map)
joined_attr_cols = [rename_map.get(c, c) for c in dam_attr_cols]

# 4) Robust matching strategy:
#    A) Prefer intersects with largest overlap area
#    B) If no intersects for a lake, fallback to nearest dam feature
left = glake.reset_index(drop=True).copy()
left['_row_id'] = np.arange(len(left))

left_geom = left[['_row_id', 'geometry']]
right_geom = dam_join[joined_attr_cols + ['geometry']]

inter = gpd.overlay(left_geom, right_geom, how='intersection', keep_geom_type=False)
if not inter.empty:
    inter['_overlap_area'] = inter.geometry.area
    # Keep one dam row per lake: max overlap area
    inter_best = inter.sort_values(['_row_id', '_overlap_area'], ascending=[True, False], kind='mergesort')
    inter_best = inter_best.drop_duplicates(subset=['_row_id'], keep='first')
    inter_best = inter_best[['_row_id'] + joined_attr_cols]
else:
    inter_best = pd.DataFrame(columns=['_row_id'] + joined_attr_cols)

# Nearest fallback for non-intersecting rows
missing_ids = set(left['_row_id']) - set(inter_best['_row_id'])
if missing_ids:
    nearest = gpd.sjoin_nearest(
        left_geom[left_geom['_row_id'].isin(missing_ids)],
        right_geom,
        how='left',
        distance_col='_dist_m'
    )
    nearest_best = nearest.sort_values(['_row_id', '_dist_m'], ascending=[True, True], kind='mergesort')
    nearest_best = nearest_best.drop_duplicates(subset=['_row_id'], keep='first')
    nearest_best = nearest_best[['_row_id'] + joined_attr_cols]
else:
    nearest_best = pd.DataFrame(columns=['_row_id'] + joined_attr_cols)

attr_map = pd.concat([inter_best, nearest_best], ignore_index=True)
attr_map = attr_map.drop_duplicates(subset=['_row_id'], keep='first').set_index('_row_id')

# 5) Assign attributes back to glake
for c in joined_attr_cols:
    glake[c] = pd.Series(attr_map[c]).reindex(range(len(glake))).values

matched_rows = glake[joined_attr_cols].notna().any(axis=1).sum() if joined_attr_cols else 0
print(f"Added {len(joined_attr_cols)} attribute column(s).")
print(f"Rows with at least one transferred attribute: {matched_rows} / {len(glake)}")

Exact geometry-key matches: 0
Spatial intersects matches: 6942
Added 6 attribute column(s).
Rows with at least one transferred attribute: 6923 / 6923


In [84]:
import rasterio

In [85]:
earthquake = rasterio.open(r'model_data\GEM-GSHM_PGA-475y-rock_v2023\v2023_1_pga_475_rock_3min.tif')
landslide = rasterio.open(r'model_data\global-landslide-susceptibility-map-2-27-23.tif')

In [92]:
# Assign earthquake/landslide susceptibility to each lake polygon
# Rule:
# 1) pick the raster pixel with maximum polygon-pixel intersection area
# 2) if equal overlap area, pick the higher pixel value

import numpy as np
import geopandas as gpd
import rasterio
from rasterio.features import geometry_window
from rasterio.windows import transform as window_transform
from rasterio.transform import xy
from shapely.geometry import box
from tqdm import tqdm


def assign_dominant_pixel_value(gdf, raster_path, out_col, band=1, area_tol=1e-12):
    if gdf is None or gdf.empty:
        raise ValueError("Input GeoDataFrame is empty.")

    with rasterio.open(raster_path) as src:
        if src.crs is None:
            raise ValueError(f"Raster has no CRS: {raster_path}")

        # Reproject polygons into raster CRS for intersection tests
        gdf_r = gdf.to_crs(src.crs).copy()

        nodata = src.nodata
        out_vals = np.full(len(gdf_r), np.nan, dtype=float)

        for i, geom in tqdm(enumerate(gdf_r.geometry), total=len(gdf_r), desc=f"Sampling {out_col}"):
            if geom is None or geom.is_empty:
                continue

            # Repair invalid geometry when possible
            if not geom.is_valid:
                geom = geom.buffer(0)
                if geom.is_empty:
                    continue

            # Use geometry-driven window (more robust than rounded bbox windows)
            try:
                win = geometry_window(src, [geom], pad_x=0, pad_y=0)
            except Exception:
                # No overlap with raster extent
                continue

            if win.width <= 0 or win.height <= 0:
                continue

            arr = src.read(band, window=win, masked=True)
            valid_mask = ~np.ma.getmaskarray(arr)
            if not valid_mask.any():
                continue

            w_transform = window_transform(win, src.transform)

            best_area = -1.0
            best_val = np.nan

            valid_rc = np.argwhere(valid_mask)
            for r, c in valid_rc:
                val = arr[r, c]

                # Skip nodata/invalid values
                if nodata is not None and np.isclose(float(val), nodata, equal_nan=True):
                    continue
                if np.isnan(float(val)):
                    continue

                # Pixel polygon in raster CRS
                x_ul, y_ul = xy(w_transform, int(r), int(c), offset='ul')
                x_lr, y_lr = xy(w_transform, int(r), int(c), offset='lr')
                pix = box(min(x_ul, x_lr), min(y_ul, y_lr), max(x_ul, x_lr), max(y_ul, y_lr))

                inter_area = geom.intersection(pix).area
                if inter_area <= 0:
                    continue

                v = float(val)
                if inter_area > best_area + area_tol:
                    best_area = inter_area
                    best_val = v
                elif abs(inter_area - best_area) <= area_tol:
                    # Tie-breaker: keep higher raster value
                    if np.isnan(best_val) or v > best_val:
                        best_val = v

            if best_area > 0:
                out_vals[i] = best_val

    out = gdf.copy()
    out[out_col] = out_vals
    return out


# --- Set your TIFF paths here ---
eq_tif = r"model_data\GEM-GSHM_PGA-475y-rock_v2023\v2023_1_pga_475_rock_3min.tif"
ls_tif = r"model_data\global-landslide-susceptibility-map-2-27-23.tif"

# --- Run for both rasters ---
glake = assign_dominant_pixel_value(glake, eq_tif, out_col="eq_susceptibility")
glake = assign_dominant_pixel_value(glake, ls_tif, out_col="ls_susceptibility")

print("Added columns: eq_susceptibility, ls_susceptibility")
print(glake[["eq_susceptibility", "ls_susceptibility"]].describe(include="all"))

Sampling ls_susceptibility: 100%|██████████| 6923/6923 [00:05<00:00, 1222.90it/s]

Added columns: eq_susceptibility, ls_susceptibility
       eq_susceptibility  ls_susceptibility
count        6923.000000        6923.000000
mean            0.257598           3.962588
std             0.086678           0.760681
min             0.095870           1.000000
25%             0.188056           4.000000
50%             0.257158           4.000000
75%             0.335432           4.000000
max             0.553538           5.000000


In [93]:
glake.to_file(r'model_data\shapefiles\glacial_lakes\glake_final_18.gpkg', driver='GPKG')

In [94]:
glake.columns

Index(['Type', 'Elevation', 'GLAKE_ID', 'area_2020', 'perimeter_2020',
       'area_1990', 'perimeter_1990', 'geometry_1990', 'area_2000',
       'perimeter_2000', 'geometry_2000', 'area_2010', 'perimeter_2010',
       'geometry_2010', 'area_2015', 'perimeter_2015', 'geometry_2015',
       'area_1990_km2', 'area_2000_km2', 'area_2010_km2', 'area_2015_km2',
       'area_2020_km2', 'expansion_rate_km2_yr', 'expansion_rate_pct_yr',
       'glof_happened', 'glof_count', 'lake_type', 'geometry',
       'distance_to_glacier_m', 'nearest_rgiid', 'is_connected', 'dam_crest_m',
       'dam_height_m', 'dam_slope_deg', 'dam_width_m', 'freeboard_m', 'wse_m',
       'dam_crest_m_dam', 'dam_height_m_dam', 'dam_slope_deg_dam',
       'dam_width_m_dam', 'freeboard_m_dam', 'wse_m_dam', 'eq_susceptibility',
       'ls_susceptibility'],
      dtype='str')

In [95]:
glaciers_gdf.columns

Index(['RGIId', 'GLIMSId', 'BgnDate', 'EndDate', 'CenLon', 'CenLat',
       'O1Region', 'O2Region', 'Area', 'Zmin', 'Zmax', 'Zmed', 'Slope',
       'Aspect', 'Lmax', 'Status', 'Connect', 'Form', 'TermType', 'Surging',
       'Linkages', 'Name', 'geometry'],
      dtype='str')

In [96]:
import geopandas as gpd
import pandas as pd

# Load your data
# glakes = gpd.read_file(r"D:\Rubel\M.Tech\MTP\Phase 2\lakes_geodatabase.gpkg")
# glaciers_gdf = gpd.read_file(r"D:\Rubel\M.Tech\MTP\Phase 2\glaciers.gpkg")  # adjust

# Only upload glaciers that are actually referenced by a lake
# (no point computing slope for glaciers with no associated lake)
referenced_ids = glake['nearest_rgiid'].dropna().unique()
glaciers_needed = glaciers_gdf[glaciers_gdf['RGIId'].isin(referenced_ids)].copy()

print(f"Total glaciers in GDF: {len(glaciers_gdf)}")
print(f"Glaciers referenced by lakes: {len(glaciers_needed)}")

# Ensure WGS84 — GEE requires it for asset upload
glaciers_needed = glaciers_needed.to_crs(epsg=4326)

# Keep only essential columns to minimise asset size
glaciers_needed = glaciers_needed[['RGIId', 'geometry']]

# Export
out_path = r"glaciers_for_gee.shp"
glaciers_needed.to_file(out_path)
print(f"Saved {len(glaciers_needed)} glaciers → {out_path}")

Total glaciers in GDF: 63620
Glaciers referenced by lakes: 4561
Saved 4561 glaciers → glaciers_for_gee.shp


In [2]:
import geopandas as gpd
glake = gpd.read_file(r'model_data\shapefiles\glacial_lakes\glake_final_19.gpkg')
glake.columns

Index(['Type', 'Elevation', 'GLAKE_ID', 'area_2020', 'perimeter_2020',
       'area_1990', 'perimeter_1990', 'geometry_1990', 'area_2000',
       'perimeter_2000', 'geometry_2000', 'area_2010', 'perimeter_2010',
       'geometry_2010', 'area_2015', 'perimeter_2015', 'geometry_2015',
       'area_1990_km2', 'area_2000_km2', 'area_2010_km2', 'area_2015_km2',
       'area_2020_km2', 'expansion_rate_km2_yr', 'expansion_rate_pct_yr',
       'glof_happened', 'glof_count', 'lake_type', 'distance_to_glacier_m',
       'nearest_rgiid', 'is_connected', 'dam_crest_m_dam', 'dam_height_m_dam',
       'dam_slope_deg_dam', 'dam_width_m_dam', 'freeboard_m_dam', 'wse_m_dam',
       'eq_susceptibility', 'ls_susceptibility', 'volume_m3', 'max_depth_m',
       'surface_elevation_m', 'area_m2', 'perimeter_m', 'geometry'],
      dtype='object')

In [4]:
columns = ['dam_crest_m', 'dam_height_m', 'dam_slope_deg', 'dam_width_m', 'freeboard_m', 'wse_m']
glake.drop(columns = columns, errors='ignore', inplace=True)
glake.columns

Index(['Type', 'Elevation', 'GLAKE_ID', 'area_2020', 'perimeter_2020',
       'area_1990', 'perimeter_1990', 'geometry_1990', 'area_2000',
       'perimeter_2000', 'geometry_2000', 'area_2010', 'perimeter_2010',
       'geometry_2010', 'area_2015', 'perimeter_2015', 'geometry_2015',
       'area_1990_km2', 'area_2000_km2', 'area_2010_km2', 'area_2015_km2',
       'area_2020_km2', 'expansion_rate_km2_yr', 'expansion_rate_pct_yr',
       'glof_happened', 'glof_count', 'lake_type', 'distance_to_glacier_m',
       'nearest_rgiid', 'is_connected', 'dam_crest_m_dam', 'dam_height_m_dam',
       'dam_slope_deg_dam', 'dam_width_m_dam', 'freeboard_m_dam', 'wse_m_dam',
       'eq_susceptibility', 'ls_susceptibility', 'volume_m3', 'max_depth_m',
       'surface_elevation_m', 'area_m2', 'perimeter_m', 'geometry'],
      dtype='object')

In [12]:
output_path = r'model_data\shapefiles\glacial_lakes\glake_final_19.gpkg'
glake.to_file(output_path, driver='GPKG')
print(f"Saved cleaned glake to: {output_path}")

Saved cleaned glake to: model_data\shapefiles\glacial_lakes\glake_final_19.gpkg


In [9]:
import pandas as pd
slope = pd.read_csv(r'hma_glacier_mean_slope1.csv')
slope.columns

Index(['RGIId', 'G11_mean_slope_deg'], dtype='object')

In [3]:
glake['nearest_rgiid']

0       RGI60-14.07524
1       RGI60-14.09122
2       RGI60-14.09057
3       RGI60-14.09028
4       RGI60-14.08932
             ...      
6918    RGI60-15.09545
6919    RGI60-15.03847
6920    RGI60-15.02371
6921    RGI60-15.10335
6922    RGI60-14.14461
Name: nearest_rgiid, Length: 6923, dtype: object

In [13]:
# glake = glake.merge(slope, left_on='nearest_rgiid', right_on='RGIId', how='left')
glake.columns

Index(['Type', 'Elevation', 'GLAKE_ID', 'area_2020', 'perimeter_2020',
       'area_1990', 'perimeter_1990', 'geometry_1990', 'area_2000',
       'perimeter_2000', 'geometry_2000', 'area_2010', 'perimeter_2010',
       'geometry_2010', 'area_2015', 'perimeter_2015', 'geometry_2015',
       'area_1990_km2', 'area_2000_km2', 'area_2010_km2', 'area_2015_km2',
       'area_2020_km2', 'expansion_rate_km2_yr', 'expansion_rate_pct_yr',
       'glof_happened', 'glof_count', 'lake_type', 'distance_to_glacier_m',
       'nearest_rgiid', 'is_connected', 'dam_crest_m_dam', 'dam_height_m_dam',
       'dam_slope_deg_dam', 'dam_width_m_dam', 'freeboard_m_dam', 'wse_m_dam',
       'eq_susceptibility', 'ls_susceptibility', 'volume_m3', 'max_depth_m',
       'surface_elevation_m', 'area_m2', 'perimeter_m', 'geometry', 'RGIId',
       'G11_mean_slope_deg'],
      dtype='object')